In [3]:
import numpy as np
import pandas as pd
import datetime as dt
import ast

In [4]:
pl1_file = pd.read_csv('/Users/mizwally/Desktop/DYAD_01-tangrams/DYAD_01_SCAN_011_abstract-images.csv')
pl2_file = pd.read_csv('/Users/mizwally/Desktop/DYAD_01-tangrams/DYAD_01_SCAN_012_abstract-images.csv')
#pl1_file = pd.read_csv('C:\\Users\\mizwa\\Downloads\\DYAD_01_SCAN_011_abstract-images.csv')
#pl2_file = pd.read_csv('C:\\Users\\mizwa\\Downloads\\DYAD_01_SCAN_012_abstract-images.csv')
pl1_ID = 'SCAN_011'
pl2_ID = 'SCAN_012'
dyad_ID = 'DYAD_01'

In [5]:
pl1_director = pl1_file[pl1_file['Role'] == 'director']
pl1_guessor = pl1_file[pl1_file['Role'] == 'guessor']
pl2_director = pl2_file[pl2_file['Role'] == 'director']
pl2_guessor = pl2_file[pl2_file['Role'] == 'guessor']

In [6]:
pl1_rounds = []
pl2_rounds = []
for i in range(len(pl2_director.index)):
    round = {'director': list(pl2_director.iloc[i][9:15]), 
             'guessor': list(pl1_guessor.iloc[i][9:15]), 
             'inputs': list(pl1_guessor.iloc[i][15:27:2]),
             'rts': list(pl1_guessor.iloc[i][16:28:2]),
             'start_time': pl2_director.iloc[i]['Round_start_time'],
             'end_time': pl2_director.iloc[i]['Round_end_time'],
             'control': pl1_guessor.iloc[i]['Control?']}
    pl1_rounds.append(round)
for i in range(len(pl1_director.index)):
    round = {'director': list(pl1_director.iloc[i][9:15]), 
             'guessor': list(pl2_guessor.iloc[i][9:15]), 
             'inputs': list(pl2_guessor.iloc[i][15:27:2]),
             'rts': list(pl2_guessor.iloc[i][16:28:2]),
             'start_time': pl1_director.iloc[i]['Round_start_time'],
             'end_time': pl1_director.iloc[i]['Round_end_time'],
             'control': pl2_guessor.iloc[i]['Control?']}
    pl2_rounds.append(round)

In [5]:
def check_answers(director, guessor, inputs, rts, start_time, diff) :
    #see when guessor image occurs in director list, then compare to answer in box
    correct = []
    control = True
    box_results = {}
    start = dt.datetime.strptime(start_time, '%H:%M:%S')
    #get accuracy, put image # and correct in slot for what box it was in
    
    for i, d in enumerate(director) :
        for g, inp, rt in zip(guessor, inputs, rts) :
            if d == g :
                control = False
                box_num = guessor.index(d)
                #calculate RT here
                #do time difference between start + 20*i-1 and rt
                #write separate rule given image display time after DYAD_09
                #calculate based on response # not image # 
                if inp == '[]' :
                    correct.append([d, None])
                    box_results[box_num] = [d, None]
                else : 
                    rt_last = ast.literal_eval(rt)[-1]
                    rt = dt.datetime.strptime(rt_last, '%H:%M:%S')
                    rt_act = rt - (start + dt.timedelta(0,(diff.total_seconds()/6)*i))
                    print('rt_dt: ', rt, 'start: ', start, 'added_time: ', (diff.total_seconds()/6)*i, 'rt: ', rt_act)
                    if i+1 == int(inp[2:1:-1]) :
                        correct.append([d, True])
                        if rt_act.total_seconds() < 0:
                            box_results[box_num] = [d, True, -1]
                        else :
                            correct.append([d, True])
                            box_results[box_num] = [d, True, rt_act.total_seconds()]
                    else :
                        if rt_act.total_seconds() < 0:
                            box_results[box_num] = [d, False, -1]
                        else :
                            correct.append([d, False])
                            box_results[box_num] = [d, False, rt_act.total_seconds()]
    print(box_results)
    if control :
        return ['control']
    else :
        #return correct
        return box_results

In [ ]:
#need to think about coding reaction time when incorrect as well, ER wants to have all answers and rts
def make_answer_key(director, guessor) :
    answer_key = {}
    control = True
    for g in guessor :
        for j, d in enumerate(director) :
            if g == d :
                control = False
                answer_key[g] = j
    if control :
        return ['control']
    else :
        return answer_key          

In [7]:
def calc_rt(start, rt, diff, i) :    
    rt = dt.datetime.strptime(rt, '%H:%M:%S')
    rt_act = rt - (start + dt.timedelta(0,(diff.total_seconds()/6)*i))
    print('rt_dt: ', rt, 'start: ', start, 'added_time: ', (diff.total_seconds()/6)*i, 'rt: ', rt_act)
    return rt_act.total_seconds()

In [ ]:
def check_correct(director, guessor, inputs, rts, start, diff) :
    #see when guessor image occurs in director list, then compare to answer in box
    graded_set = []
    #get accuracy, put image # and correct in slot 
    
    #get accuracy % and round rt
    
    for i, d in enumerate(director) :
        accurate = False
        correct_indices = []
        correct_rt = []
        all_incorrect_responses = []
        all_incorrect_rt = []
        for j, g, in enumerate(guessor) :
            incorrect_indices = []
            incorrect_responses = []
            incorrect_rt = []
            inp = ast.literal_eval(inputs[j])
            if str(i+1) in inp :
                if d == g :      
                    correct_indices = [k for k, x in enumerate(inp) if x == str(i+1)]    
                    incorrect_responses = [x for x in inp if x != str(i+1)]  
                    print(inp, i+1, 'correct', correct_indices)              
                    crt_set = [ast.literal_eval(rts[j])[k] for k in correct_indices]
                    for rt in crt_set :
                        correct_rt.append(calc_rt(start, rt, diff, i))                                     
                    if correct_indices[-1] == len(inp)-1 :
                        accurate = True 
                    print(correct_indices, accurate) 
                else :
                    incorrect_indices = [k for k, x in enumerate(inp) if x == str(i+1)]    
                    print('incorrect', inp, i+1, incorrect_indices)
                    irt_set = [ast.literal_eval(rts[j])[k] for k in incorrect_indices]
                    for ind, rt in zip(incorrect_indices, irt_set) :
                        incorrect_rt.append(calc_rt(start, rt, diff, ind)) 
            [all_incorrect_rt.append(x) for x in incorrect_rt]    
            [all_incorrect_responses.append(x) for x in incorrect_responses]        
        graded = {'image': d, 'correct?': accurate, 'correct rt': correct_rt, 'incorrect rt': all_incorrect_rt,
                  'incorrect resp box': all_incorrect_responses, 'number resp box': len(correct_rt)+len(all_incorrect_responses)}
        print('graded: ', graded)
        graded_set.append(graded)        
    return graded_set       

In [83]:
pl1_answers = []
pl2_answers = []
for i in range(len(pl1_rounds)) :
    pl1_st = dt.datetime.strptime(pl1_rounds[i]['start_time'], '%H:%M:%S')
    pl1_end = dt.datetime.strptime(pl1_rounds[i]['end_time'], '%H:%M:%S')
    pl1_diff = pl1_end - pl1_st
    pl2_st = dt.datetime.strptime(pl2_rounds[i]['start_time'], '%H:%M:%S')
    pl2_end = dt.datetime.strptime(pl2_rounds[i]['end_time'], '%H:%M:%S')
    pl2_diff = pl2_end - pl2_st
    #print(i, pl1_st, pl1_end, pl1_diff, pl1_rounds[i]['control'])
    if pl1_rounds[i]['control'] == False :
        pl1_ans = check_correct(pl1_rounds[i]['director'], pl1_rounds[i]['guessor'], pl1_rounds[i]['inputs'], 
                                pl1_rounds[i]['rts'], pl1_st, pl1_diff)
        pl1_answers.append(pl1_ans)
        #print(pl1_ans)
    else :
        pl1_answers.append(['control'])
    '''   
    if pl2_rounds[i]['control'] == False :
        pl2_ans = check_correct(pl2_rounds[i]['director'], pl2_rounds[i]['guessor'], pl2_rounds[i]['inputs'], 
                                pl2_rounds[i]['rts'], pl2_st, pl2_diff)
        pl2_answers.append(pl2_ans)
    else :
        pl2_answers.append(['control'])
    ''' 
#pair accuracy with reaction time and add condition to answer file

['1'] 1 correct [0]
rt_dt:  1900-01-01 17:33:50 start:  1900-01-01 17:33:44 added_time:  0.0 rt:  0:00:06
[0] True
graded:  {'image': '48.png', 'correct?': True, 'correct rt': [6.0], 'incorrect rt': [], 'incorrect resp box': [], 'number resp box': 1}
['2'] 2 correct [0]
rt_dt:  1900-01-01 17:34:10 start:  1900-01-01 17:33:44 added_time:  20.166666666666668 rt:  0:00:05.833333
[0] True
graded:  {'image': '34.png', 'correct?': True, 'correct rt': [5.833333], 'incorrect rt': [], 'incorrect resp box': [], 'number resp box': 1}
['3'] 3 correct [0]
rt_dt:  1900-01-01 17:34:30 start:  1900-01-01 17:33:44 added_time:  40.333333333333336 rt:  0:00:05.666667
[0] True
graded:  {'image': '19.png', 'correct?': True, 'correct rt': [5.666667], 'incorrect rt': [], 'incorrect resp box': [], 'number resp box': 1}
['4'] 4 correct [0]
rt_dt:  1900-01-01 17:34:51 start:  1900-01-01 17:33:44 added_time:  60.5 rt:  0:00:06.500000
[0] True
graded:  {'image': '22.png', 'correct?': True, 'correct rt': [6.5], 'i

In [73]:
pl1_answer_file = pd.concat([pl1_guessor[pl1_guessor.columns[:9]].reset_index(drop=True), pd.DataFrame(pl1_answers)], axis=1).drop(columns=['Role', 'round_duration', 'completion_status', 'Control?'])
#pl1_answer_file.drop(columns=['Folder', 'Role', 'Round_start_time', 'Round_end_time', 'round_duration', 'completion_stsatus'])

In [74]:
pl1_answer_file

,Block,Round,Folder,Round_start_time,Round_end_time,0,1,2,3,4,5
0,1,2,easy1,17:33:44,17:35:44,"{'image': '48.png', 'accurate?': True, 'correc...","{'image': '34.png', 'accurate?': True, 'correc...","{'image': '19.png', 'accurate?': True, 'correc...","{'image': '22.png', 'accurate?': True, 'correc...","{'image': '24.png', 'accurate?': True, 'correc...","{'image': '44.png', 'accurate?': True, 'correc..."
1,2,1,hard1,17:35:59,17:37:59,"{'image': '30.png', 'accurate?': True, 'correc...","{'image': '31.png', 'accurate?': False, 'corre...","{'image': '23.png', 'accurate?': False, 'corre...","{'image': '6.png', 'accurate?': False, 'correc...","{'image': '13.png', 'accurate?': False, 'corre...","{'image': '32.png', 'accurate?': False, 'corre..."
2,3,2,control1,17:42:46,17:44:46,control,None,None,None,None,None
3,4,1,easy2,17:45:01,17:47:01,"{'image': '10.png', 'accurate?': True, 'correc...","{'image': '21.png', 'accurate?': True, 'correc...","{'image': '42.png', 'accurate?': True, 'correc...","{'image': '54.png', 'accurate?': True, 'correc...","{'image': '7.png', 'accurate?': True, 'correct...","{'image': '11.png', 'accurate?': True, 'correc..."
4,5,2,control2,17:51:48,17:53:48,control,None,None,None,None,None
5,6,2,hard2,17:56:19,17:58:19,"{'image': '57.png', 'accurate?': True, 'correc...","{'image': '41.png', 'accurate?': True, 'correc...","{'image': '53.png', 'accurate?': True, 'correc...","{'image': '35.png', 'accurate?': True, 'correc...","{'image': '58.png', 'accurate?': True, 'correc...","{'image': '36.png', 'accurate?': True, 'correc..."


In [75]:
pl1_answer_file.to_csv('/Users/mizwally/Desktop/DYAD_01-tangrams/DYAD_01_SCAN_011_answers.csv', index=False)